#### This notebook is used to create dataframes (i.e. df_avg_sim_{arch}_epoch{epoch}.pkl) holding results related to "Attention-Head Stability" for the architecture. 
#### We create such dataframes for all 26 architectures.
#### Eventually, all those 26 dataframes are used in "most_and_least_stable_layers.ipynb" to create plots for Section 4.6 (Most- and least-stable layers)

In [ ]:
import gc
import itertools
import math
import os
import random
import sys
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Callable, Literal, TypeAlias

import einops
import numpy as np
import pandas as pd
import torch as t
from datasets import load_dataset
from IPython.display import clear_output, display
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table

from transformer_lens import HookedTransformer, HookedTransformerConfig
from tabulate import tabulate
from torch import Tensor, nn
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformer_lens import ActivationCache, loading_from_pretrained
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, to_numpy
from transformer_lens import utils

import matplotlib.pyplot as plt
import seaborn as sns
import torch

from scipy.sparse import csr_array
from scipy.sparse.csgraph import maximum_bipartite_matching, min_weight_full_bipartite_matching

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

## Prompt Sentences Data

In [ ]:
import pickle

prompts_file = "100_prompts"

with open(f"../{prompts_file}.pkl", "rb") as f:
    prompts = pickle.load(f)

print(f"Loaded {len(prompts)} prompts from ./{prompts_file}.pkl")

Loaded 100 prompts from ./100_prompts.pkl


## Model & State_dict Loading 

In [ ]:
import importlib

# ------------------- Specify Model Arch -------------------
arch = "l8_h8"

# ------------------- Load Model Config -------------------

def load_named_config(module_name: str, config_name: str) -> dict:
    """
    Import a module that defines CONFIGS: Dict[str, Dict[str, Any]]
    and return CONFIGS[config_name].
    """
    try:
        mod = importlib.import_module(module_name)
    except Exception as e:
        raise ImportError(f"Could not import config module '{module_name}': {e}") from e

    if not hasattr(mod, "CONFIGS"):
        raise AttributeError(f"Module '{module_name}' does not define CONFIGS.")

    CONFIGS = getattr(mod, "CONFIGS")
    if config_name not in CONFIGS:
        available = ", ".join(sorted(CONFIGS.keys()))
        raise KeyError(f"Config '{config_name}' not found in {module_name}. Available: {available}")

    return dict(CONFIGS[config_name])  # copy so we can tweak

cfg_dict = load_named_config("model_configs", arch)


# ------------------- Model Configuration -------------------

# Build HookedTransformerConfig using the loaded config
cfg = HookedTransformerConfig(
    n_layers=cfg_dict["n_layers"],
    d_model=cfg_dict["d_model"],
    n_heads=cfg_dict["n_heads"],
    d_head=cfg_dict["d_head"],
    d_mlp=cfg_dict.get("d_mlp", None),
    n_ctx=cfg_dict["n_ctx"],
    act_fn=cfg_dict.get("act_fn", "gelu"),
    d_vocab=cfg_dict["d_vocab"],
    init_weights=True,
    tokenizer_name=cfg_dict["tokenizer_name"],
    model_name=cfg_dict.get("model_name", arch),
    attn_only=cfg_dict.get("attn_only", False),
)

### Run next cell to cover each variant of architecture in "arch_list" creating a pickle file for it. 
### Skip this cell if pickle files have been already created.

In [ ]:
import os

# Setting device to CPU as GPU memory is insufficient for this computation, but for smaller number of prompts/models it can be set to GPU
device = 'cpu'

# Run for each arch but only for the single mentioned epoch (first entry of epoch_list).
# Then plot all arch curves on the same plot.
arch_list = [f"{arch}", f"{arch}_wd"]

cfg_dict = load_named_config("model_configs",arch)

# ------------------- Model Configuration -------------------

# Build HookedTransformerConfig using the loaded config
cfg = HookedTransformerConfig(
    n_layers=cfg_dict["n_layers"],
    d_model=cfg_dict["d_model"],
    n_heads=cfg_dict["n_heads"],
    d_head=cfg_dict["d_head"],
    d_mlp=cfg_dict.get("d_mlp", None),
    n_ctx=cfg_dict["n_ctx"],
    act_fn=cfg_dict.get("act_fn", "gelu"),
    d_vocab=cfg_dict["d_vocab"],
    init_weights=True,
    tokenizer_name=cfg_dict["tokenizer_name"],
    attn_only=cfg_dict.get("attn_only", False),
)


# constants reused from your original cell
SEEDS = [i for i in range(1, 51)]
if "gpt2" in arch_list[0]:
    SEEDS = [i for i in range(1, 6)] 

avg_sim_dir = "df/avg_sim"
# Constants
SCRATCH = "Path to root directory"
NUM_LAYERS = cfg.n_layers
NUM_HEADS = cfg.n_heads
ATTN_ONLY = cfg.attn_only
chkpt_file = "final.pt"
shard = 9
epoch = 1

# helper (kept from original)
def lower_triang(mat):
    lower_triangular_mat = torch.tril(mat)
    mask = torch.tril(torch.ones_like(mat, device=device)).bool()
    lower_triangular_vec = lower_triangular_mat[mask]
    return lower_triangular_vec

cos = t.nn.CosineSimilarity(dim=1, eps=1e-08)

# storage across architectures for combined plotting
arch_layer_means = {}
arch_layer_vars = {}

for arch in arch_list:

    print(f"Processing arch: {arch}")
    cfg_dict = load_named_config("model_configs", arch)

    # Build HookedTransformerConfig for this arch
    cfg = HookedTransformerConfig(
        n_layers=cfg_dict["n_layers"],
        d_model=cfg_dict["d_model"],
        n_heads=cfg_dict["n_heads"],
        d_head=cfg_dict["d_head"],
        d_mlp=cfg_dict.get("d_mlp", None),
        n_ctx=cfg_dict["n_ctx"],
        act_fn=cfg_dict.get("act_fn", "gelu"),
        d_vocab=cfg_dict["d_vocab"],
        init_weights=True,
        tokenizer_name=cfg_dict["tokenizer_name"],
        model_name=cfg_dict.get("model_name", arch),
        attn_only=cfg_dict.get("attn_only", False),
    )

    chkpt_dir = SCRATCH + "chkpts/" + arch

    # Load models for this epoch
    models = []
    for SEED in SEEDS:
        cfg.seed = SEED
        cfg.init_weights = True
        model = HookedTransformer(cfg)
        models.append(model)

    for ind, SEED in enumerate(SEEDS):
        if (arch == "gpt2") or (arch == "gpt2_wd"):
            model_state_dict = t.load(chkpt_dir + f"/gpt2_seed{SEED}_shard{shard}_epoch{epoch}_owt/{chkpt_file}")
            models[ind].load_and_process_state_dict(model_state_dict, fold_ln=False)
        else:
            if ATTN_ONLY:
                model_state_dict = t.load(
                    chkpt_dir + f"/causal_attn_only_l{NUM_LAYERS}_h{NUM_HEADS}_seed{SEED}_epoch{epoch}_c4_gelu/{chkpt_file}"
                )
                models[ind].load_and_process_state_dict(model_state_dict, fold_ln=False)

            else:
                model_state_dict = t.load(
                    chkpt_dir + f"/causal_attn_l{NUM_LAYERS}_h{NUM_HEADS}_seed{SEED}_epoch{epoch}_c4_gelu/{chkpt_file}"
                )
                models[ind].load_and_process_state_dict(model_state_dict, fold_ln=False)

    # Setting device to CPU as GPU memory is insufficient for this computation, but for smaller number of prompts/models it can be set to GPU
    device = 'cpu'

    # run prompts to collect caches (using CPU to avoid CUDA OOM)
    prompts_cache = []
    for prompt in prompts:
        cache_for_prompt = []
        for ind in range(len(SEEDS)):
            _, cache_i = models[ind].run_with_cache(prompt, remove_batch_dim=True)
            # Keep cache on CPU
            cache_i = cache_i.to('cpu')
            cache_for_prompt.append(cache_i)
        prompts_cache.append(cache_for_prompt)

    # compute cosine-similarity matrix across layers/heads/models per prompt
    NUM_MODELS = len(models)
    NUM_HEADS = models[0].cfg.n_heads
    NUM_LAYERS = models[0].cfg.n_layers
    NUM_PROMPTS = len(prompts_cache)

    # Set the list of anchors index that you want to cover.
    # Default is [0]
    anchor_range = 1

    # Free memory held by models
    del models
    del model_state_dict
    torch.cuda.empty_cache()

    all_results = []

    for anchor in range(anchor_range):

        print("Computing cosine similarity matrix...")
        prompts_cs_matrix = t.empty(( NUM_LAYERS, NUM_HEADS, NUM_MODELS, NUM_HEADS, NUM_PROMPTS), device=device)

        # Compute cosine-similarity matrix across layers/heads/models per prompt
        for ind_prompt in range(NUM_PROMPTS):
            cache = prompts_cache[ind_prompt]
            cache_anchor = cache[anchor].to(device)
            
            for layer in range(NUM_LAYERS):
                for head_anchor in range(NUM_HEADS):
                        
                        head_anchor_attn = cache_anchor[utils.get_act_name('pattern',layer,'a')][head_anchor]
                        head_anchor_attn = lower_triang(head_anchor_attn.to(device))
                        head_anchor_attn = t.unsqueeze(head_anchor_attn, 0)

                        for model_i in range(NUM_MODELS):
                            cache_model_i = cache[model_i].to(device)
                            for head_pair in range(NUM_HEADS):
                                head_i_attn = cache_model_i[utils.get_act_name('pattern',layer,'a')][head_pair]
                                head_i_attn = lower_triang(head_i_attn.to(device))
                                head_i_attn = t.unsqueeze(head_i_attn, 0)
                                cos_score_i = cos(head_anchor_attn, head_i_attn)
                                prompts_cs_matrix[layer, head_anchor, model_i, head_pair, ind_prompt] = cos_score_i
                                del head_i_attn, cos_score_i

                            del cache_model_i

                        del head_anchor_attn

            del cache_anchor
            torch.cuda.empty_cache()


                    
        # aggregate to get per-(layer, head_anchor) cos_sim (same aggregation as original)
        #results = []
        for layer_i in range(NUM_LAYERS):

            csh = prompts_cs_matrix[layer_i,:,:,:]                # [head_anchor, model_i, head_pair, prompts]
            csh = torch.mean(csh, -1)                            # mean over prompts -> [head_anchor, model_i, head_pair]
            csh = csh.max(dim=-1, keepdim=True)                  # max over head_pair -> values shape [head_anchor, model_i, 1]
            csh = csh.values[:,:,0]                              # [head_anchor, model_i]
            csh = torch.cat([csh[:, :anchor], csh[:, anchor+1:]], dim=1) # remove comparison of anchor model with itself
            csh = csh.mean(dim=-1).reshape(NUM_HEADS, 1)         # mean over models -> [head_anchor, 1]

            for head_anchor in range(NUM_HEADS):
                all_results.append({
                    "arch": arch,
                    "epoch": epoch,
                    "anchor_idx": anchor,
                    "layer": layer_i + 1,
                    "head_anchor": head_anchor + 1,
                    "cos_sim": csh[head_anchor].item()
                })
                    

    # create dataframe with per-anchor results and also store aggregated mean/var per (layer, head)
    df_all_anchors = pd.DataFrame(all_results)
    # ensure directory exists
    out_dir = os.path.join(SCRATCH, avg_sim_dir)
    os.makedirs(out_dir, exist_ok=True)

    # save detailed per-anchor dataframe for this arch
    out_path = os.path.join(out_dir, f"df_avg_sim_{arch}_epoch{epoch}.pkl")
    df_all_anchors.to_pickle(out_path)
    print(f"Saved df_all_anchors to {out_path}")
